# FAISS Ad-hoc Query Playground
Set the paths below, run the setup cell once, then re-run the query cell whenever you want to test a different prompt.

In [2]:
import json
from pathlib import Path
from typing import List

import numpy as np
from sentence_transformers import SentenceTransformer

from udlf_text_espresso.retrieval.dense import get_faiss

# --- Basic configuration (edit these for your run) ---
RUN_ID = "paper-assets"
DATASET_NAME = "scidocs"
INDEX_ALIAS = "miniLM-L6-v2"  # matches dense_indexing.model_alias
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cpu"  # change to "cuda" or "mps" if available
TOP_K = 10

# If you launch this notebook outside the project root, point OUTPUTS_ROOT to the absolute folder.
OUTPUTS_ROOT = Path("/Users/luisvenezian/Documents/masters/udlf-text-espresso/outputs")
if not OUTPUTS_ROOT.exists():
    raise FileNotFoundError(f"Configured OUTPUTS_ROOT does not exist: {OUTPUTS_ROOT}")

# --- Locate artifacts ---
run_root = OUTPUTS_ROOT / RUN_ID / "dataset" / DATASET_NAME
index_dir = run_root / "indexes" / INDEX_ALIAS
faiss_path = index_dir / "faiss.index"
docids_path = index_dir / "docids.json"

if not faiss_path.exists():
    raise FileNotFoundError(f"Missing FAISS index at {faiss_path}")
if not docids_path.exists():
    raise FileNotFoundError(f"Missing docids.json at {docids_path}")

docids_payload = json.loads(docids_path.read_text(encoding="utf-8"))
DOC_IDS: List[str]
if isinstance(docids_payload, dict) and "doc_ids" in docids_payload:
    DOC_IDS = list(docids_payload["doc_ids"])
elif isinstance(docids_payload, list):
    DOC_IDS = [str(x) for x in docids_payload]
else:
    raise ValueError(f"Unexpected docids format in {docids_path}")

faiss = get_faiss()
index = faiss.read_index(str(faiss_path))
encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)

print(f"Ready. Index size={len(DOC_IDS):,} docs (dimension={index.d}).")

Ready. Index size=25,657 docs (dimension=384).


In [11]:
# 78495383450e02c5fe817e408726134b3084905d        
QUERY_TEXT = "A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect"

query_vec = encoder.encode(
    [QUERY_TEXT],
    convert_to_numpy=True,
    normalize_embeddings=True,
)
query_vec = np.asarray(query_vec, dtype=np.float32)

scores, indices = index.search(query_vec, TOP_K)

# Load documents
documents_path = run_root / "transformed" / "documents" / "data.jsonl"
doc_map = {}
with open(documents_path, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        doc_map[doc['id']] = doc

print(f"Query: {QUERY_TEXT}\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    if idx < 0 or idx >= len(DOC_IDS):
        continue
    doc_id = DOC_IDS[idx]
    print(f"#{rank:02d} | score={score:.4f} | doc_id={doc_id}")
    
    if doc_id in doc_map:
        doc = doc_map[doc_id]
        print(f"     Text: {doc.get('contents', 'N/A')[:100]}...")
    print()

Query: A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect

#01 | score=0.5769 | doc_id=86e87db2dab958f1bd5877dc7d5b8105d6e31e46
     Text: A Hybrid EP and SQP for Dynamic Economic Dispatch with Nonsmooth Fuel Cost Function
Dynamic economic...

#02 | score=0.4477 | doc_id=cd31ecb3b58d1ec0d8b6e196bddb71dd6a921b6d
     Text: Economic dispatch for a microgrid considering renewable energy cost functions
Microgrids are operate...

#03 | score=0.4378 | doc_id=1f376c10b20319121102db78e7790cf47d8fa046
     Text: Optimal Reconfiguration for Supply Restoration With Informed A$^{\ast}$  Search
Reconfiguration of r...

#04 | score=0.4209 | doc_id=91de962e115bcf65eaf8579471a818ba8c5b0ea6
     Text: Cuckoo search algorithm: a metaheuristic approach to solve structural optimization problems
In this ...

#05 | score=0.4119 | doc_id=3fd46ca896d023df8c8af2b3951730d8c38defdd
     Text: Training neural nets with the reactive tabu search
In this paper the task of training subs